# 003 · Bitwise Operators

> **Syntax › Operators** · Python 3.14

Python has the same six bitwise operators as C#. What differs is the integers they work on: Python's `int` has **arbitrary precision** and no fixed width, so there are no unsigned types, no overflow, and no `>>>`.

**In this notebook you will learn:**

- the bitwise operators `&`, `|`, `^`, `~`, `<<`, `>>`
- how arbitrary-precision integers change the picture (`~x == -x - 1`, no unsigned types)
- how to emulate fixed-width and unsigned arithmetic with masks
- converting to and from binary/hex with `bin()`, `hex()`, `format()` and `int(text, base)`
- using integers as bit flags, and counting bits with `int.bit_count()`

## C# mental model

| C# | Python | Note |
|---|---|---|
| `a & b`, `a \| b`, `a ^ b` | `a & b`, `a \| b`, `a ^ b` | same |
| `~a` | `~a` | always `-a - 1` (no fixed width) |
| `a << n`, `a >> n` | `a << n`, `a >> n` | never overflow; `>>` is arithmetic (sign-preserving) |
| `a >>> n` (C# 11) | — | no logical shift; mask first: `(a & 0xFFFFFFFF) >> n` |
| `uint`, `ulong`, `byte` | — | only one `int` type; use masks |
| `0b1010`, `0xFF`, `1_000` | `0b1010`, `0xFF`, `1_000` | same literals (plus `0o17` for octal) |
| `Convert.ToString(x, 2)` | `bin(x)` / `format(x, "b")` | |
| `Convert.ToInt32("1010", 2)` | `int("1010", 2)` | any base from 2 to 36 |
| `BitOperations.PopCount(x)` | `x.bit_count()` | |
| `[Flags] enum` | `enum.IntFlag` | see *Classes & OOP › 009 · Enums* |

## The operators

| Operator | Meaning | Example (`a = 0b1100`, `b = 0b1010`) |
|---|---|---|
| `&` | AND — 1 where both bits are 1 | `0b1000` |
| `\|` | OR — 1 where either bit is 1 | `0b1110` |
| `^` | XOR — 1 where the bits differ | `0b0110` |
| `~` | NOT — invert every bit | `-13` |
| `<<` | shift left (multiply by 2ⁿ) | `a << 2` → `0b110000` |
| `>>` | shift right (floor-divide by 2ⁿ) | `a >> 2` → `0b11` |

In [ ]:
a = 0b1100   # 12
b = 0b1010   # 10

print(f"a & b  = {a & b:04b}")
print(f"a | b  = {a | b:04b}")
print(f"a ^ b  = {a ^ b:04b}")
print(f"a << 2 = {a << 2:b}")
print(f"a >> 2 = {a >> 2:b}")
print(f"~a     = {~a}")

The `:04b` inside the f-strings is a format spec meaning "binary, zero-padded to 4 digits" *(see Data Types › 003 · String Formatting)*. Augmented forms `&=`, `|=`, `^=`, `<<=`, `>>=` also exist.

## Arbitrary precision: no overflow, no fixed width

A C# `int` is exactly 32 bits, so shifting left eventually loses bits — and the shift count is even masked to 5 bits, so `1 << 32` is `1`. A Python `int` just grows:

```csharp
// C#
Console.WriteLine(1 << 31);    // -2147483648 (overflow into the sign bit)
Console.WriteLine(1 << 32);    // 1  (shift count is taken modulo 32!)
Console.WriteLine(1L << 40);   // 1099511627776
```

In [ ]:
print(1 << 31)     # 2147483648 — no sign-bit overflow
print(1 << 32)     # 4294967296
print(1 << 100)    # still exact
print((1 << 100).bit_length())   # 101 bits needed

## `~x == -x - 1`

Conceptually, Python treats a negative integer as if it had an **infinite** number of leading 1-bits in two's complement. Inverting all bits of a non-negative number therefore gives a negative one, and the identity `~x == -x - 1` always holds. In C# you'd see the same *values* for a signed `int`, but for `uint`/`byte` the result depends on the width:

```csharp
// C#
Console.WriteLine(~5);            // -6
Console.WriteLine(~5u);           // 4294967290  (uint)
Console.WriteLine((byte)~5);      // 250
```

In [ ]:
for x in [0, 5, -6, 255]:
    print(f"{'~' + str(x):>5} = {~x:>5}    -x - 1 = {-x - 1:>5}")

To get the C# `uint`/`byte` result, **mask** the value to the width you want — `& 0xFF` keeps the low 8 bits, `& 0xFFFFFFFF` the low 32:

In [ ]:
print(~5 & 0xFF)          # 250 — like (byte)~5
print(~5 & 0xFFFFFFFF)    # 4294967290 — like ~5u

## Shifting right: arithmetic only, no `>>>`

Python's `>>` on a negative number keeps the sign — it is a floor division by 2ⁿ, like C#'s `>>` on a signed `int`. There is no unsigned shift operator `>>>` (added in C# 11), because there is no fixed width to shift zeros into. Mask to the desired width first:

```csharp
// C#
Console.WriteLine(-16 >> 2);    // -4
Console.WriteLine(-16 >>> 28);  // 15  (logical shift on a 32-bit int)
```

In [ ]:
print(-16 >> 2)                    # -4 (same as -16 // 4)
print((-16 & 0xFFFFFFFF) >> 28)    # 15 — emulates C#'s -16 >>> 28

⚠️ **Gotcha:** when you port bit-twiddling code from C# (hashes, checksums, protocol parsing), every place that relied on 32-bit wrap-around must be masked explicitly, e.g. `h = (h * 31 + c) & 0xFFFFFFFF`. Otherwise the numbers just keep growing.

In [ ]:
def hash32(text):
    """A simple 32-bit string hash, as you might write it in C# with uint arithmetic."""
    h = 0
    for ch in text:
        h = (h * 31 + ord(ch)) & 0xFFFFFFFF   # mask = emulate uint overflow
    return h

print(hash32("hello world"))

## Binary, hex and octal: conversions

`bin()`, `hex()` and `oct()` produce prefixed strings; `format()` (or an f-string spec) gives you control over prefix and padding. `int(text, base)` parses the other way.

```csharp
// C#
Console.WriteLine(Convert.ToString(42, 2));        // 101010
Console.WriteLine(42.ToString("X"));               // 2A
Console.WriteLine(Convert.ToInt32("101010", 2));   // 42
Console.WriteLine(Convert.ToInt32("2A", 16));      // 42
```

In [ ]:
n = 42
print(bin(n), hex(n), oct(n))                  # 0b101010 0x2a 0o52
print(format(n, "b"), format(n, "08b"), format(n, "X"), f"{n:#06x}")

print(int("101010", 2), int("2A", 16), int("52", 8))
print(int("0b101010", 0))   # base 0: detect the base from the prefix
print(0b101010 == 0x2A == 0o52 == 42)

⚠️ **Gotcha:** `bin()` of a negative number shows a minus sign, not the two's-complement bit pattern. Mask first to see the bits as C# would:

In [ ]:
print(bin(-5))                    # -0b101
print(format(-5 & 0xFF, "08b"))   # 11111011 — the 8-bit two's-complement pattern

## Bit flags

Before reaching for an enum, the classic pattern is plain integer constants combined with `|`, tested with `&` and cleared with `& ~`:

```csharp
// C#
[Flags]
enum Permission { None = 0, Read = 1, Write = 2, Execute = 4 }

var p = Permission.Read | Permission.Write;
bool canWrite = (p & Permission.Write) != 0;    // or p.HasFlag(Permission.Write)
p &= ~Permission.Write;                          // clear a flag
```

In [ ]:
READ = 1 << 0      # 0b001
WRITE = 1 << 1     # 0b010
EXECUTE = 1 << 2   # 0b100

perms = READ | WRITE
print(f"{perms:03b}", "can write:", bool(perms & WRITE))

perms &= ~WRITE    # clear a flag
print(f"{perms:03b}", "can write:", bool(perms & WRITE))

perms ^= EXECUTE   # toggle a flag
print(f"{perms:03b}", "can execute:", bool(perms & EXECUTE))

In real code, prefer `enum.IntFlag` (or `enum.Flag`) — the direct counterpart of a `[Flags] enum`, with readable names and the `in` operator for testing *(see Classes & OOP › 009 · Enums)*. A quick preview:

In [ ]:
from enum import IntFlag

class Permission(IntFlag):
    READ = 1
    WRITE = 2
    EXECUTE = 4

p = Permission.READ | Permission.WRITE
print(repr(p))                     # <Permission.READ|WRITE: 3>
print(Permission.WRITE in p, int(p))   # True 3

## Counting bits

`int.bit_count()` returns the number of 1-bits (the *population count*) — like `BitOperations.PopCount` in C#. `int.bit_length()` returns how many bits are needed to represent the number (ignoring sign), like `64 - BitOperations.LeadingZeroCount(x)` for positive values.

In [ ]:
n = 0b1011_0110
print(n.bit_count())        # 5
print(n.bit_length())       # 8
print((-n).bit_count())     # 5 — counts the bits of abs(n)

def is_power_of_two(x):
    return x > 0 and x & (x - 1) == 0    # classic trick: clears the lowest set bit

print([x for x in range(1, 70) if is_power_of_two(x)])

⚠️ **Gotcha:** precedence differs from C#. In Python, `&`, `^` and `|` bind **more tightly** than comparisons; in C# they bind **more loosely**. So `x & 1 == 0` means `(x & 1) == 0` in Python, whereas in C# it means `x & (1 == 0)` — a compile error for an `int`.

In [ ]:
x = 6
print(x & 1 == 0)       # True — parsed as (x & 1) == 0
print((x & 1) == 0)     # True — same, but explicit
print(x & (1 == 0))     # 0    — the C# reading (int & bool is legal in Python, since bool is an int)

The flip side: `x > 0 & x < 10` does **not** mean "between 0 and 10" in Python — the `&` grabs `0 & x` first *(see Operators › 002 · Comparison and Logical Operators)*. Use `and` for logic, and add parentheses around bitwise sub-expressions so readers don't need to remember the table.

## Bonus: the same operators on other types

The operators are overloadable, and several built-in types use them with a set-like meaning:

- `bool`: `True & False` → `False`, `True | False` → `True` (the result stays a `bool`, but it doesn't short-circuit — use `and`/`or` for logic)
- `set`: `|` union, `&` intersection, `^` symmetric difference *(see Collections › 004 · Sets)*
- `dict`: `|` merges two dicts *(see Collections › 003 · Dictionaries)*

In [ ]:
print(True & False, True | False, True ^ True)
print({1, 2, 3} & {2, 3, 4}, {1, 2} | {3}, {"a": 1} | {"b": 2})

## Summary

| Task | C# | Python |
|---|---|---|
| AND / OR / XOR / NOT | `& \| ^ ~` | `& \| ^ ~` |
| Shift | `<< >>` | `<< >>` (no overflow) |
| Unsigned shift | `>>>` | `(x & 0xFFFFFFFF) >> n` |
| Fixed-width wrap | automatic (`uint`, `byte`) | mask: `x & 0xFF` |
| To binary / hex text | `Convert.ToString(x, 2)` / `x.ToString("X")` | `bin(x)`, `hex(x)`, `format(x, "08b")` |
| Parse binary / hex | `Convert.ToInt32(s, 2)` | `int(s, 2)`, `int(s, 16)` |
| Pop count | `BitOperations.PopCount(x)` | `x.bit_count()` |
| Flags | `[Flags] enum` | `enum.IntFlag` (or int constants) |

## Exercises

1. Write `set_bit(value, n)`, `clear_bit(value, n)` and `is_set(value, n)` that set, clear and test bit `n` of an integer.
2. Emulate a C# `byte`: write `to_byte(x)` so that `to_byte(300) == 44` and `to_byte(-1) == 255`.
3. Parse the hex colour `"1E90FF"` into its red, green and blue components (`30, 144, 255`) using `int(..., 16)`, shifts and masks.
4. Count how many numbers between 0 and 255 have exactly four 1-bits.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def set_bit(value, n):
    return value | (1 << n)

def clear_bit(value, n):
    return value & ~(1 << n)

def is_set(value, n):
    return (value >> n) & 1 == 1

v = set_bit(0, 3)
print(bin(v), is_set(v, 3), bin(clear_bit(v, 3)))   # 0b1000 True 0b0

# 2 — masking to 8 bits gives the same wrap-around as an unchecked C# cast
def to_byte(x):
    return x & 0xFF

print(to_byte(300), to_byte(-1))   # 44 255

# 3
rgb = int("1E90FF", 16)
red = (rgb >> 16) & 0xFF
green = (rgb >> 8) & 0xFF
blue = rgb & 0xFF
print(red, green, blue)            # 30 144 255

# 4
print(sum(1 for n in range(256) if n.bit_count() == 4))   # 70
```

</details>